<a href="https://colab.research.google.com/github/hannahchen63/veritasai/blob/main/notebooks/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Results log

### Layer 1: pro-ED detection (300 test tweets: 70 harmful, 230 not)
| Run | Data | Missed harmful (FN) | Harmless blurred (FP) |
|---|---|---|---|
| Keyword filter | - | 49/70 | 25/230 |
| TF-IDF + LogReg | tweets | 9/70 | 13/230 |
| Run 1: DistilRoBERTa | tweets | 7/70 | 12/230 |
| Run 2: DistilRoBERTa | cleaned tweets | 9/70 | 14/230 |
| Run 3: DistilRoBERTa | cleaned tweets + 400 recipes | 6/70 | 19/230 |
| Run 4: DistilRoBERTa | cleaned tweets + 400 recipes + 50 normal Instagram posts | 9/70 | 14/230 |

Recipes wrongly blurred (Run 3): 0/100 (Run 2: 71/100)
Normal Instagram posts wrongly blurred by layer 1: Run 3 = 11/75 (~15%), Run 4 = 1/25 held out (4%)
Chosen layer 1 model: Run 4 (threshold 0.4)

### Layer 2: diet culture (119 hand-labeled Instagram-style posts: 44 diet, 75 not; 5-fold CV)
| Method | Missed diet (FN) | Normal blurred (FP) | Notes |
|---|---|---|---|
| Diet keyword filter | 1/44 | 3/75 | list written after seeing data |
| Zero-shot (DeBERTa) | 23/44 | 3/75 | |
| Fine-tuned from layer 1 | 6/44 | 9/75 | |
| Embeddings + LogReg | 6/44 | 4/75 | misses exact calorie numbers |

All methods wrongly blur W007 and W011 (pro-recovery posts).

Labeling: team changed 8/126 rule-based labels; dropped C026 (empty), duplicates N017/D020,
and image-only posts C008, C016, C019, R009.

Next: fresh test set (new searches, never seen) to compare embeddings vs keywords fairly;
add pro-recovery and keyword-free diet examples to training.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/jabenitez88/NLP-EatingDisordersBERT/main/tweets_labelled_2000.csv"
ed = pd.read_csv(URL, sep=";")
ed = ed[["text_orig", "ProED"]].rename(columns={"text_orig": "text", "ProED": "label"})
print(ed.shape)
print(ed["label"].value_counts())

(2000, 2)
label
0    1533
1     467
Name: count, dtype: int64


Load the NLP-EatingDisordersBERT dataset from github

Keep two columns: the text and the label

For labels, 1 = promotes eating disorder, 0 = does not

In [2]:
from sklearn.model_selection import train_test_split
import re

def clean(text):
    text = re.sub(r"^RT\s+", "", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"http\S+", "", text)
    return re.sub(r"\s+", " ", text).strip()

ed = ed.dropna()
ed["text"] = ed["text"].astype(str).map(clean)

train, temp = train_test_split(ed, test_size=0.3, stratify=ed["label"], random_state=42)
val, test   = train_test_split(temp, test_size=0.5, stratify=temp["label"], random_state=42)
print(len(train), len(val), len(test))   # 1400 300 300

1400 300 300


Split the dataset into training, validation, and testing

Stratify keeps about 23% of harmful tweets in each group

Random_state=42 makes the split the same everytime you run it

In [3]:
from sklearn.metrics import confusion_matrix

def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"{name}")
    print(f"  missed harmful posts (false negatives): {fn}/{fn+tp}")
    print(f"  harmless posts blurred (false positives): {fp}/{fp+tn}")
    print(f"  accuracy: {(tp+tn)/len(y_true):.2f}\n")

Reports predictions to their true labels

Calculates false negatives (missing harmful posts)

Calculates false positives (harmless posts get blurred)

In [4]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. dumb baselines
report("Always show", test.label, [0]*len(test))
report("Always blur", test.label, [1]*len(test))

# 2. keyword filter: what a simple non-AI extension would do
KEYWORDS = ["calorie", "calories", "kcal", "diet", "fasting", "fast",
            "skinny", "thin", "weight", "fat", "lose", "starve"]
def keyword_filter(text):
    return int(any(re.search(rf"\b{k}\b", text.lower()) for k in KEYWORDS))
report("Keyword filter", test.label, test.text.map(keyword_filter))

# 3. classic ML: TF-IDF + logistic regression
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(vec.fit_transform(train.text), train.label)
report("TF-IDF + LogReg", test.label, clf.predict(vec.transform(test.text)))

Always show
  missed harmful posts (false negatives): 70/70
  harmless posts blurred (false positives): 0/230
  accuracy: 0.77

Always blur
  missed harmful posts (false negatives): 0/70
  harmless posts blurred (false positives): 230/230
  accuracy: 0.23

Keyword filter
  missed harmful posts (false negatives): 49/70
  harmless posts blurred (false positives): 25/230
  accuracy: 0.75

TF-IDF + LogReg
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 13/230
  accuracy: 0.93



Baselines for our AI to beat

Always show and always blur are extremes

The keyword filter only blurs a test if it has a certain word like "calories" or "diet"

TF-IDF + LogReg is a classic ML that turns tweets into weighted word counts and learns which words lean harmful (does not learn context of the full tweet)

In [5]:
!pip install -q datasets
from datasets import load_dataset

rec = load_dataset("AWeirdDev/all-recipes-xs", split="train").to_pandas()
rec["text"] = (rec["name"].astype(str) + ". " + rec["review"].astype(str)).str.slice(0, 280)
rec = rec[["text"]].drop_duplicates()
rec["label"] = 0
print(len(rec), "recipes")
print(rec.text.head(3).tolist())   # check they look reasonable

# 400 recipes go into training, the rest are held out as a recipe test set
rec_train = rec.sample(400, random_state=42)
rec_test  = rec.drop(rec_train.index)

train_mixed = pd.concat([train, rec_train], ignore_index=True).sample(frac=1, random_state=42)
print(train_mixed.label.value_counts())

README.md:   0%|          | 0.00/3.50k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  324kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/500 [00:00<?, ? examples/s]

500 recipes
['Dutch Oven Crunchy Corned Beef. I found this recipe attached to a 6-pack of beer on St. Patrick\'s Day in 1995.\nI\'ve been serving it as a holiday tradition since. Even self-proclaimed corned beef haters love this corned beef. A typical comment is, "could be the best corned beef I\'', 'Spicy Spicy Ranch Dressing. This spicy ranch dip with hot sauce and cayenne is excellent served with fries, poppers, and even wings.', 'Amazing Oven-Roasted Potatoes. These oven-roasted potatoes are golden on the outside and tender on the inside. Made with minimum oil for a less greasy option than regular fried potatoes!']
label
0    1473
1     327
Name: count, dtype: int64


Downloads AWeirdDev/all-recipes-xs dataset from HuggingFace

Combines recipe's name and short description into one text, under 280 characters

Labels every recipe as 0 (harmless)

Splits the recipe with 400 in training and the rest as a test set the model never sees

Adds the 400 training to the original training tweets to make a mixed dataset (train_mixed) and shuffles it

In [6]:
# Run 4: add normal Instagram-style posts to layer 1 training as "not pro-ED"
FEED_URL = "https://raw.githubusercontent.com/hannahchen63/veritasai/main/feed_dataset_paraphrased%20-%20feed_dataset_paraphrased.csv"
feed_raw = pd.read_csv(FEED_URL)
feed_raw = feed_raw[feed_raw["text"].notna() & (feed_raw["text"].str.strip() != "")]
feed_raw = feed_raw[~feed_raw["sample_id"].isin(["N017", "D020", "C008", "C016", "C019", "R009"])]

normal = feed_raw[feed_raw["diet_culture"] == 0][["sample_id", "text"]].copy()
normal["label"] = 0
normal_train = normal.sample(50, random_state=42)
normal_test  = normal.drop(normal_train.index)          # 25 held out

train_mixed = pd.concat([train, rec_train, normal_train[["text", "label"]]],
                        ignore_index=True).sample(frac=1, random_state=42)
print(len(train_mixed), "training examples")             # should be 1850

1850 training examples


Add normal Instagram style posts to layer 1 training

In [7]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)   # should say cuda

MODEL_NAME = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_loader(df, shuffle):
    enc = tokenizer(list(df.text), truncation=True, padding="max_length",
                    max_length=128, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"],
                       torch.tensor(df.label.values))
    return DataLoader(ds, batch_size=16, shuffle=shuffle)

train_loader = make_loader(train_mixed, True)
rec_test_loader = make_loader(rec_test, False)
print("Training on", len(train_loader.dataset), "examples")
val_loader   = make_loader(val, False)
test_loader  = make_loader(test, False)

Using: cuda


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Training on 1850 examples


Loads DistilRoBERTa-base model

Tokenizer turns tweets into numbers

DataLoader gives tweets in batch size 16

cuda is colab gpu

In [8]:
torch.manual_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# class weights: harmful posts are rarer (about 1 in 4), so missing one costs more
n0, n1 = (train_mixed.label == 0).sum(), (train_mixed.label == 1).sum()
weights = torch.tensor([1.0, n0 / n1], dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

def get_probs(loader):
    model.eval(); probs = []
    with torch.no_grad():
        for ids, mask, _ in loader:
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.array(probs)

for epoch in range(4):
    model.train(); total = 0
    for ids, mask, labels in train_loader:
        optimizer.zero_grad()
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        loss = loss_fn(logits, labels.to(device))
        loss.backward(); optimizer.step(); total += loss.item()
    val_pred = (get_probs(val_loader) >= 0.5).astype(int)
    print(f"Epoch {epoch+1}: train loss {total/len(train_loader):.3f}")
    report(f"  validation", val.label, val_pred)

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1: train loss 0.319
  validation
  missed harmful posts (false negatives): 11/70
  harmless posts blurred (false positives): 16/230
  accuracy: 0.91

Epoch 2: train loss 0.150
  validation
  missed harmful posts (false negatives): 7/70
  harmless posts blurred (false positives): 24/230
  accuracy: 0.90

Epoch 3: train loss 0.109
  validation
  missed harmful posts (false negatives): 13/70
  harmless posts blurred (false positives): 8/230
  accuracy: 0.93

Epoch 4: train loss 0.067
  validation
  missed harmful posts (false negatives): 14/70
  harmless posts blurred (false positives): 10/230
  accuracy: 0.92



Adds "harmful or not" layer to the model

Trains 1,400 tweets with 4 epochs (4 passes through the data)

Class weights means that harmful posts are rare (1 out of 4) so making a mistake on them costs 3x more to make the model not lazy

Loss goes down from 0.345 to 0.080

After each epoch, it checks the validation set

In [9]:
val_probs = get_probs(val_loader)
print("threshold | missed harmful (FN) | harmless blurred (FP)")
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(val.label, p, labels=[0, 1]).ravel()
    print(f"   {t:.1f}    |        {fn}/{fn+tp}         |      {fp}/{fp+tn}")

threshold | missed harmful (FN) | harmless blurred (FP)
   0.2    |        11/70         |      14/230
   0.3    |        12/70         |      14/230
   0.4    |        12/70         |      13/230
   0.5    |        14/70         |      10/230
   0.6    |        14/70         |      8/230
   0.7    |        16/70         |      6/230


Model outputs score from 0 to 1 for "how harmful" text is

Threshold decides cutoff for blurring

Low threshold = blur more, fewer harmful posts get through, but more harmless posts get blurred wrongly

In [10]:
THRESHOLD = 0.4

In [11]:
test_pred = (get_probs(test_loader) >= THRESHOLD).astype(int)
report(f"DistilRoBERTa (threshold {THRESHOLD})", test.label, test_pred)

DistilRoBERTa (threshold 0.4)
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 14/230
  accuracy: 0.92



Runs trained model on test data set it has never seen

Results shown

Run 1, without cleaning twitter features:

DistilRoBERTa (threshold 0.4)
  missed harmful posts (false negatives): 7/70
  harmless posts blurred (false positives): 12/230
  accuracy: 0.94

Run 2, with cleaning twitter only (no recipes yet)

DistilRoBERTa (threshold 0.4)
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 14/230
  accuracy: 0.92

In [12]:
test_pred = (get_probs(test_loader) >= THRESHOLD).astype(int)
report("Mixed model - original test", test.label, test_pred)

rec_pred = (get_probs(rec_test_loader) >= THRESHOLD).astype(int)
print(f"Mixed model - recipes wrongly blurred: {rec_pred.sum()}/{len(rec_test)}")

# keyword filter and TF-IDF on the same recipes, for comparison
print(f"Keyword filter - recipes blurred: {rec_test.text.map(keyword_filter).sum()}/{len(rec_test)}")
print(f"TF-IDF - recipes blurred: {clf.predict(vec.transform(rec_test.text)).sum()}/{len(rec_test)}")

Mixed model - original test
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 14/230
  accuracy: 0.92

Mixed model - recipes wrongly blurred: 0/100
Keyword filter - recipes blurred: 2/100
TF-IDF - recipes blurred: 0/100


Runs new model on original set to make sure adding recipes does not harm the detection

Counts how many recipes the the new model wrongly blurs

Runs keyword filter and TF-IDF on the same recipes for comparison

That was layer 1, layer 2 begins here

In [13]:
model.save_pretrained("ed_model")
tokenizer.save_pretrained("ed_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('ed_model/tokenizer_config.json', 'ed_model/tokenizer.json')

In [14]:
from sklearn.model_selection import StratifiedKFold

feed = pd.read_csv("https://raw.githubusercontent.com/hannahchen63/veritasai/main/feed_dataset_paraphrased%20-%20feed_dataset_paraphrased.csv")
feed = feed[feed["text"].notna() & (feed["text"].str.strip() != "")]   # drops C026 (empty)
feed = feed[~feed["sample_id"].isin(["N017", "D020"])]                 # drop duplicate copies
IMAGE_ONLY = ["C008", "C016", "C019", "R009"]                           # signal only in image/on-screen text
image_only = feed[feed["sample_id"].isin(IMAGE_ONLY)].copy()
feed = feed[~feed["sample_id"].isin(IMAGE_ONLY)].reset_index(drop=True)
feed["label"] = feed["diet_culture"].astype(int)
image_only["label"] = image_only["diet_culture"].astype(int)
print(len(feed), "posts:", feed.label.value_counts().to_dict())   # 119 posts: {0: 75, 1: 44}

def report_dc(name, y, p, ids=None):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    print(f"{name}\n  missed diet-culture posts (FN): {fn}/{fn+tp}\n  normal posts blurred (FP): {fp}/{fp+tn}\n  accuracy: {(tp+tn)/len(y):.2f}")
    if ids is not None:
        y, p = np.array(y), np.array(p)
        print("  missed:", list(ids[(y == 1) & (p == 0)]), "| wrongly blurred:", list(ids[(y == 0) & (p == 1)]))
    print()

119 posts: {0: 75, 1: 44}


In [15]:
DIET = r"\bcals?\b|calori|kcal|deficit|macros?\b|weight ?loss|lose weight|losing weight|\blost\b|fat ?loss|\blbs\b|\bkg\b|\bdiet\b|low ?cal|skinny|\blean\b|snatched"
kw_pred = feed.text.str.lower().str.contains(DIET).astype(int)
report_dc("Diet keyword filter", feed.label, kw_pred, feed.sample_id)

Diet keyword filter
  missed diet-culture posts (FN): 1/44
  normal posts blurred (FP): 3/75
  accuracy: 0.97
  missed: ['N011'] | wrongly blurred: ['W007', 'W011', 'D001']



In [16]:
from transformers import pipeline
zs = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0", device=0)
ZS_LABELS = ["diet culture: calorie counting, weight loss or body image",
             "a regular food or recipe post"]
zs_out = zs(list(feed.text), ZS_LABELS, hypothesis_template="This post is {}.")
zs_pred = np.array([int(o["labels"][0] == ZS_LABELS[0]) for o in zs_out])
report_dc("Zero-shot model", feed.label, zs_pred, feed.sample_id)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Zero-shot model
  missed diet-culture posts (FN): 23/44
  normal posts blurred (FP): 3/75
  accuracy: 0.78
  missed: ['R002', 'R003', 'R004', 'R005', 'R007', 'R010', 'N003', 'N011', 'D003', 'D006', 'D014', 'D015', 'D016', 'C002', 'C003', 'C007', 'C010', 'C011', 'C015', 'C017', 'C022', 'C024', 'C025'] | wrongly blurred: ['W007', 'W009', 'W011']



In [17]:
def predict(m, df):
    m.eval(); probs = []
    with torch.no_grad():
        for ids, mask, _ in make_loader(df, False):
            logits = m(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.array(probs)

def train_layer2(train_df, epochs=6):
    torch.manual_seed(42)
    m = AutoModelForSequenceClassification.from_pretrained("ed_model").to(device)   # start from layer 1
    opt = torch.optim.AdamW(m.parameters(), lr=3e-5)
    n0, n1 = (train_df.label == 0).sum(), (train_df.label == 1).sum()
    lf = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, n0 / n1], dtype=torch.float).to(device))
    for _ in range(epochs):
        m.train()
        for ids, mask, labels in make_loader(train_df, True):
            opt.zero_grad()
            loss = lf(m(input_ids=ids.to(device), attention_mask=mask.to(device)).logits, labels.to(device))
            loss.backward(); opt.step()
    return m

extra_recipes = rec_train.sample(100, random_state=42)[["text", "label"]]   # extra "fine" examples, training only
oof = np.zeros(len(feed))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr, te) in enumerate(skf.split(feed, feed.label)):
    train_df = pd.concat([feed.iloc[tr][["text", "label"]], extra_recipes], ignore_index=True)
    m2 = train_layer2(train_df)
    oof[te] = predict(m2, feed.iloc[te])
    print(f"fold {fold+1} done")
    del m2; torch.cuda.empty_cache()

l2_pred = (oof >= 0.5).astype(int)
report_dc("Fine-tuned layer 2 (5-fold CV)", feed.label, l2_pred, feed.sample_id)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

fold 1 done


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

fold 2 done


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

fold 3 done


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

fold 4 done


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

fold 5 done
Fine-tuned layer 2 (5-fold CV)
  missed diet-culture posts (FN): 6/44
  normal posts blurred (FP): 6/75
  accuracy: 0.90
  missed: ['R007', 'N011', 'W021', 'D015', 'D016', 'C002'] | wrongly blurred: ['W007', 'W011', 'W018', 'D001', 'D013', 'D019']



In [18]:
final_df = pd.concat([feed[["text", "label"]], extra_recipes], ignore_index=True)
diet_model = train_layer2(final_df)
print("image-only posts caught:", (predict(diet_model, image_only) >= 0.5).sum(), "/", len(image_only))
diet_model.save_pretrained("diet_model"); tokenizer.save_pretrained("diet_model")
!zip -r -q diet_model.zip diet_model
from google.colab import files; files.download("diet_model.zip")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

image-only posts caught: 0 / 4


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import cross_val_predict

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
X = embedder.encode(list(feed.text), normalize_embeddings=True)

clf_emb = LogisticRegression(class_weight="balanced", max_iter=2000, C=2.0)
emb_pred = cross_val_predict(clf_emb, X, feed.label, cv=skf)   # same 5 folds as before
report_dc("Embeddings + LogReg (5-fold CV)", feed.label, emb_pred, feed.sample_id)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings + LogReg (5-fold CV)
  missed diet-culture posts (FN): 6/44
  normal posts blurred (FP): 4/75
  accuracy: 0.92
  missed: ['N011', 'D003', 'D006', 'D015', 'D016', 'C025'] | wrongly blurred: ['W007', 'W011', 'D001', 'D017']



In [20]:
l1_probs = predict(AutoModelForSequenceClassification.from_pretrained("ed_model").to(device), feed)
feed["l1_flag"] = (l1_probs >= THRESHOLD).astype(int)
print(pd.crosstab(feed["diet_culture"], feed["l1_flag"],
                  rownames=["diet culture label"], colnames=["layer 1 blurs?"]))
print("layer 1 blurs these normal posts:", list(feed[(feed.l1_flag == 1) & (feed.label == 0)].sample_id))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

layer 1 blurs?       0  1
diet culture label       
0                   74  1
1                   44  0
layer 1 blurs these normal posts: ['W015']


In [21]:
nt_pred = (get_probs(make_loader(normal_test, False)) >= THRESHOLD).astype(int)
print(f"Held-out normal Instagram posts wrongly blurred: {nt_pred.sum()}/{len(normal_test)}")

Held-out normal Instagram posts wrongly blurred: 1/25


In [22]:
import joblib
final_emb_clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=2.0).fit(X, feed.label)
joblib.dump(final_emb_clf, "diet_clf.joblib")
from google.colab import files
files.download("diet_clf.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import os
if not os.path.exists("ed_model"):          # save it first if it isn't saved yet
    model.save_pretrained("ed_model")
    tokenizer.save_pretrained("ed_model")
!zip -r -q ed_model.zip ed_model
from google.colab import files
files.download("ed_model.zip")



zip error: Interrupted (aborting)


FileNotFoundError: Cannot find file: ed_model.zip

In [26]:
from google.colab import drive
drive.mount("/content/drive")                # click through the sign-in popup
!cp ed_model.zip "/content/drive/MyDrive/"
!cp diet_model.zip "/content/drive/MyDrive/"

Mounted at /content/drive
cp: cannot stat 'ed_model.zip': No such file or directory


In [27]:
import joblib
from sklearn.linear_model import LogisticRegression

# train the final layer 2 classifier on all 119 posts
final_emb_clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=2.0).fit(X, feed.label)

# save the embedding model (vectorizer) + classifier together in one folder
embedder.save("diet_model")
joblib.dump(final_emb_clf, "diet_model/diet_clf.joblib")

# zip it and download
!zip -r -q diet_model.zip diet_model
from google.colab import files
files.download("diet_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>